# USalign Structure Alignment

This notebook demonstrates pairwise structure alignment using USalign. We align two structures provided as PDB text and inspect the TM-scores, which measure topological similarity for both monomers and multi-chain assemblies.

In [9]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("usalign")
display_overview("usalign")
display_docs_section("usalign", "Background")

# USalign

> USalign (Universal Structure alignment) extends TMalign to support monomers, multimers, and nucleic acid structures (`usalign-alignment`). It aligns two macromolecular structures using multimer-aware mode (`-mm 1 -ter 1`) and returns [TM-scores](https://en.wikipedia.org/wiki/Template_modeling_score) normalized by each structure's length. USalign is a compiled C++ binary that runs on CPU with no external dependencies.

## Background

**What does this tool measure/predict?**
USalign computes the TM-score for pairs of macromolecular structures, generalizing the TMalign algorithm to handle:

* **Monomeric proteins**: Standard pairwise alignment (equivalent to TMalign)
* **Multimeric complexes**: Joint alignment considering all chains, with optimal chain permutation for symmetric assemblies
* **Nucleic acid structures**: RNA and DNA structure comparison using nucleotide-aware scoring
* **Mixed complexes**: Protein-nucleic acid complexes aligned as unified structures

**Why is this important?**

* Complex structure validation: verify predicted multimer assemblies match experimental structures
* Protein design: evaluate how well designed complexes match target architectures
* RNA structure comparison: compare predicted RNA 3D structures to known folds
* Structural genomics: classify and compare protein complexes at the fold level

**Scientific foundation:**
USalign unifies multiple structure alignment methods (TMalign, MMalign, RNAalign) into a single universal framework. For multimers, it uses an iterative chain mapping algorithm that finds the optimal correspondence between chains in the two structures, then computes TM-score over all aligned chains jointly. The TM-score threshold of 0.5 for fold similarity applies equally to monomers and multimers. For nucleic acids, backbone phosphorus atoms replace C-alpha atoms as alignment anchors.

## Available tools

In [10]:
display_available_tools("usalign")

- **`run_usalign()`** — Universal structure alignment using USalign (Zhang et al., 2022). Supports monomers and multimers. Returns TM-scores normalized by each structure.

### `run_usalign`

USalign performs universal structure alignment that handles monomers, multimeric complexes, nucleic acids, and mixed protein-nucleic acid assemblies. It accepts two structures as raw PDB text, automatically identifies chains and molecular types, computes an optimal chain-to-chain mapping, and returns two TM-scores normalized by the length of each structure. Using it with multimer flags makes it the recommended tool when inputs may contain multiple chains or non-protein molecules.

In [11]:
from pathlib import Path

from proto_tools import USalignConfig, USalignInput, USalignOutput, run_usalign
from proto_tools.tools.structure_alignment.usalign.usalign import example_input

In [12]:
# Display input docs
display_api_reference("usalign", "input", "run_usalign")

# Use the tool's canonical example input — a PDB aligned against itself,
# so TM-scores should come back as 1.0.
inputs = example_input()

**Input** — `USalignInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `pdb_text_1` | `string` | required | Raw PDB content of the first structure (query / candidate). |
| `pdb_text_2` | `string` | required | Raw PDB content of the second structure (reference / target). |

In [13]:
# Display config docs
display_api_reference("usalign", "config", "run_usalign")

# USalignConfig has no tool-specific parameters; all defaults are used
config = USalignConfig()

**Config** — `USalignConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [14]:
# Run the alignment
result = run_usalign(inputs, config)

Running run_usalign [00:00]

In [15]:
# Display output docs and inspect results
display_api_reference("usalign", "output", "run_usalign")

print(f"TM-score (norm by Structure 1): {result.tm_score_structure_1:.3f}")
print(f"TM-score (norm by Structure 2): {result.tm_score_structure_2:.3f}")
print(f"Execution time: {result.execution_time:.2f}s")

**Output** — `USalignOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `tm_score_structure_1` | `number` | required | TM-score normalized by the length of Structure 1 (query). |
| `tm_score_structure_2` | `number` | required | TM-score normalized by the length of Structure 2 (reference). |

TM-score (norm by Structure 1): 1.000
TM-score (norm by Structure 2): 1.000
Execution time: 0.04s


## Export Results

Alignment results can be exported to JSON format for downstream analysis or integration with other computational pipelines.

In [16]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("usalign_result", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'usalign_result.json'}")

Exported to example_output/usalign_result.json
